#### Student Name: Kumiko Komori
#### Student ID: A18547845

*AI Usage: Used Claude for all coding and conceptual portions of this assignment.*

## Speech Formants with Linear Predictive Coding, Vocoder (Mister Blue Sky)

Instructions: 

* This notebook is an interactive assignment; please read and follow the instructions in each cell. 

* Cells that require your input (in the form of code or written response) will have 'Question #' above.

* After completing the assignment, please submit this notebook and its pdf printout and all sound files. 

## Speech Formants and LPC

In this section, you will synthesize vowel sounds, and investigate the frequencies in vowels from your own voice. 

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from numpy.random import multinomial as randm
from numpy import where
import scipy.signal as si
import matplotlib.pyplot as plt
from matplotlib import patches
import IPython.display as ipd
import librosa
import scipy
import librosa.display as ld
from scipy.io import wavfile as wavfile 
import copy


Fdict = {
    'mystery_1':[[328, 2208, 2885],[27,80,575]],
    'mystery_2':[[504, 868, 2654],[62,   108,  299]],
    'mystery_3':[[700, 1220, 2600],[130,   70,  160]]
    } # Formant frequencies in Hz

def excitation(f0, jitt, dur, nharm=None, unvoiced=False, sample_rate=None):
    sample_rate = fs if sample_rate is None else sample_rate
    nsamps = int(round(sample_rate * dur))
    assert nsamps > 0

    if unvoiced:
        sig = np.random.normal(size=nsamps)
    else:
        if nharm is None:
            nharm = int((sample_rate / 2) // f0)
        n = np.arange(nsamps)
        phase_jitter = np.random.uniform(size=nsamps) * 2 * np.pi
        omega0 = 2 * np.pi * f0 / sample_rate
        harmonics = np.arange(1, nharm)[:, None]
        sig = np.cos(
            harmonics * omega0 * n + jitt * phase_jitter
        ).sum(axis=0)

    peak = np.max(np.abs(sig))
    assert peak > 0
    return sig / peak

def voca(sig, F, Fb, sample_rate=None):
    sample_rate = fs if sample_rate is None else sample_rate
    F = np.asarray(F, dtype=float)
    Fb = np.asarray(Fb, dtype=float)

    radii = np.exp(-np.pi * Fb / sample_rate)
    angles = 2 * np.pi * F / sample_rate
    poles = radii * np.exp(1j * angles)
    B, A = si.zpk2tf(
        [],
        np.concatenate([poles, np.conj(poles)]),
        1.0,
    )

    speech = si.lfilter(B, A, np.asarray(sig, dtype=float))
    scale = np.std(speech)
    assert scale > 0
    return speech / scale, B, A

fs = 8192 # 22050  % Sampling rate in Hz ("telephone quality" for speed)

vowels = list(Fdict.keys())
f0 = 150 # Pitch in Hz
dur = 1 #one second in duration
ji = 0.1 #0.1
ex = excitation(f0,ji,dur)

text = ['mystery_1','mystery_2','mystery_3']

speech = np.zeros(1)
for t in text:
    F = np.array(Fdict[t][0])
    Fb = np.array(Fdict[t][1])
    print(t)

    vow,B,A = voca(ex,F,Fb)
    speech = np.concatenate((speech,vow))

speech = speech/np.std(speech)
plt.plot(speech)

In [ ]:
ipd.Audio(speech, rate=fs) 

##### Question 1 (10 points)

Based on the audio output, what vowels were synthesized as mystery_1, mystery_2, and mystery_3? 
Please specify using a word; for example, if you heard an 'oo' sound as in 'hoot', you may answer with the word "hoot". 

A vowel's identity is mostly set by its first two formant frequencies, **F1** and **F2** (the first two numbers in each `Fdict` entry). The rough rule:

- **F1** tracks how open my mouth is / how low my tongue is. Low F1 → tongue high/mouth closed; high F1 → tongue low/mouth open.
- **F2** tracks how far forward my tongue is. High F2 → tongue front; low F2 → tongue back.

**mystery_1 → "ee" as in "heed":** F1 = 328, F2 = 2208. F1 is very low (tongue high, mouth nearly closed) and F2 is very high (tongue pushed forward) — a high, front vowel. The textbook /i/ sits around F1 ≈ 270, F2 ≈ 2290, which matches almost exactly.

**mystery_1: heed ("ee")**

**mystery_2 → "oh" as in "boat":** F1 = 504 (mid, mouth partly open), F2 = 868 (very low, tongue pulled back) — a back, rounded vowel. Reference /o/ ("boat") is about F1 ≈ 500, F2 ≈ 910, basically a direct hit. Its neighbor /ɔ/ ("bought," F1 ≈ 570, F2 ≈ 840) is close too, so match whichever you heard.

**mystery_2: boat ("oh"), possibly bought ("aw")**

**mystery_3 → "uh" as in "cut":** F1 = 700 (mouth wide open), F2 = 1220 (central tongue) — an open, central vowel. Closest to /ʌ/ ("cut," F1 ≈ 640, F2 ≈ 1190), with /ɑ/ ("hot"/"father," F1 ≈ 730, F2 ≈ 1090) as runner-up.

**mystery_3: cut ("uh"), possibly hot ("ah")**

*Caveat:* reading vowels straight off formant numbers gets you into the right neighborhood every time, but neighboring vowels (boat vs. bought, cut vs. hot) can be genuinely close, so the audio is the tie-breaker.

Now we will examine just one vowel in greater detail. 
Select one mystery vowel to analyze below: 

In [ ]:
%matplotlib inline  

### Modify the line below:
text = ['mystery_1']

speech = np.zeros(1)
for t in text:
    F = np.array(Fdict[t][0])
    Fb = np.array(Fdict[t][1])
    print(t)

    vow,B,A = voca(ex,F,Fb)
    speech = np.concatenate((speech,vow))

speech = speech/np.std(speech)
plt.plot(speech)

In [ ]:
# Plot the power spectral density (PSD)
plt.psd(speech, 1024)
plt.show()

In [ ]:
lpc_order = 10
s = speech

a = librosa.core.lpc(s, order=lpc_order)
print(a)
s_hat = scipy.signal.lfilter([0] + -1*a[1:], [1], s)
s_err = s[1:] - s_hat[:-1]
plt.plot(s[1:])
plt.plot(s_hat[:-1], linestyle='--')
plt.legend(['y', 'y_hat'])
plt.title('LP Model Forward Prediction')
plt.show()

In [ ]:
plt.plot(s[1:101])
plt.plot(s_hat[:100])
plt.legend(['y', 'y_hat'])
plt.show()

##### Question 2 (10 points)

What is being visualized as y and y_hat on the above plot?

This plot shows **the real signal versus what the LPC model predicts the signal should be.**

Linear Predictive Coding assumes each speech sample can be estimated as a **weighted sum of the samples right before it**. With `lpc_order = 10`, the model looks back at the previous 10 samples and guesses the current one:

$$\hat{s}[n] \approx -a_1 s[n-1] - a_2 s[n-2] - \cdots - a_{10}\,s[n-10]$$

The printed `a` array is exactly those weights (the LPC coefficients). The line `s_hat = scipy.signal.lfilter(...)` just runs that "predict each sample from the previous 10" formula across the whole signal.

So on the plot:

- **`y` is the true speech signal** — the actual sample values `s` (ground truth).
- **`y_hat` is the model's prediction** of that same signal — for every point in time, the LPC filter's best guess at the current sample using only past samples. (That's why the title says "forward prediction.")

The two lines sit almost on top of each other (very clear in the zoomed-in 100-sample plot) because speech is **highly correlated** from one sample to the next — neighboring samples follow a smooth pattern set by the vocal tract, which is exactly what a linear predictor captures well.

The small gap where they don't match is the **prediction error** (the `s_err` variable). That leftover is the part of the signal the vocal-tract filter can't explain on its own — conceptually, the excitation/source (the "buzz" from the vocal folds) driving the filter. So this plot is really a visual proof that LPC models speech well.

In [ ]:
w,h = si.freqz(b=1,a = a, fs=1)
plt.plot(w,20*np.log10(h))

In [ ]:
z,p,k = si.tf2zpk(B,A)
    
unit_circle = patches.Circle((0,0), radius=1, fill=False, color='black', ls='solid', alpha=0.9)
ax = plt.subplot(111)
ax.add_patch(unit_circle)
ax.spines['left'].set_position('center')
ax.spines['bottom'].set_position('center')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
r = 1.5; plt.axis('scaled'); plt.axis([-r, r, -r, r])
ticks = [-1, -.5, .5, 1]; plt.xticks(ticks); plt.yticks(ticks)    
    
plt.plot(z.real, z.imag, 'ko', fillstyle='none', ms = 10)
plt.plot(p.real, p.imag, 'kx', fillstyle='none', ms = 10)

In [ ]:
D = np.abs(librosa.stft(s,n_fft=256,hop_length = 64))
ld.specshow(librosa.amplitude_to_db(D))

##### Question 3 (20 points)

Record yourself speaking the same vowel sound you analyzed above. 
Graph the power spectral density (PSD) of your recording alongside the PSD of the synthetic signal. 

In [ ]:
### Your Code Here
# Record yourself saying the same vowel (mystery_1) and save it next to this
# notebook, then update the filename below.
vowel_rec, vowel_sr = librosa.load("kmk_mystery_1.wav", sr=None, mono=True)

plt.figure()
plt.psd(vowel_rec, NFFT=1024, Fs=vowel_sr)
plt.psd(speech,    NFFT=1024, Fs=fs)
plt.legend(['recorded', 'synthetic'])
plt.title('PSD: recorded vs synthetic vowel (mystery_1)')
plt.xlabel('Frequency (Hz)')
plt.show()

##### Question 4 (10 points)

How does the power spectral density of your recorded signal compare to the LPC spectrum? 

Short version: **they show the same formant peaks in the same places, but the LPC spectrum is a clean smoothed outline while the recorded PSD is the full, bumpy reality.**

What each plot actually is:

- The **LPC spectrum** (the `freqz` plot) is the frequency response of the all-pole filter the model fit. With only a handful of poles it can produce just a few smooth humps, and those humps sit right at the **formants** (the vocal-tract resonances). It's essentially the *envelope* — the overall shape, fine detail thrown away on purpose.
- The **recorded PSD** is the measured energy-vs-frequency of my actual recording. It contains *everything*: the vocal-tract resonances, plus the source that drove them (the harmonic "comb" from my pitch, showing up as closely spaced spikes), plus recording noise.

Comparing them:

1. **The big peaks line up.** The broad energy bumps in my recorded PSD fall at roughly the same low frequencies as the peaks in the LPC spectrum — confirmation that my recording contains the same vowel resonances the LPC model captured.
2. **The recorded PSD is much bumpier / more detailed.** It has all the harmonic spikes and noise wiggles; the LPC curve glides smoothly through the middle of them, like the line you'd draw over the top of the recorded spectrum's peaks.
3. **The recorded signal spreads energy over a much wider bandwidth** (out to high frequencies, then dropping to a noise floor), whereas the LPC spectrum rolls off smoothly with just its couple of resonant humps.

Takeaway: LPC keeps the **filter** part of speech (the formant envelope) and discards the **source** part (harmonics and noise), which is why the LPC curve looks like a cleaned-up skeleton of the real recorded spectrum.

# Simple Singing Vocoder

In this section we will use a spoken sound to process an excitation that plays a melody. In music such an effect is known as vocoding and it is used to produce a talking musical instrument. 

In [ ]:
def lpc_to_formants(lpc, sr):
    roots = np.roots(lpc)
    roots = roots[np.imag(roots) > 0]
    roots = np.where(
        np.abs(roots) > 1,
        1 / np.conj(roots),
        roots,
    )

    freqs = np.angle(roots) * sr / (2 * np.pi)
    bws = -sr * np.log(np.abs(roots)) / np.pi

    valid = (
        np.isfinite(freqs)
        & np.isfinite(bws)
        & (freqs > 0)
        & (freqs < sr / 2)
        & (bws > 0)
    )
    order = np.argsort(freqs[valid])
    return freqs[valid][order], bws[valid][order]
    """Convert LPC to formants    
    """
        
    # extract roots, get angle and radius
    roots = np.roots(lpc)
    
    pos_roots = roots[np.imag(roots)>=0]
    if len(pos_roots)<len(roots)//2:
        pos_roots = list(pos_roots) + [0] * (len(roots)//2 - len(pos_roots))
    if len(pos_roots)>len(roots)//2:
        pos_roots = pos_roots[:len(roots)//2]
    
    w = np.angle(pos_roots)
    a = np.abs(pos_roots)
    
    order = np.argsort(w)
    w = w[order]
    a = a[order]
    
    freqs = w * (sr/(2*np.pi))
    bws =  -0.5 * (sr/(2*np.pi)) * np.log(a)    
    
    # exclude DC and sr/2 frequencies
    return freqs, bws

##### Questioon 5 [10 points] 

Record yourself speaking slowly the sentence "Mister Blue Sky". Plot a spectrogram of the speech sound.

In [ ]:
# Your code here
# Record yourself slowly saying "Mister Blue Sky" and save it (mono) next to this
# notebook, then update the filename below.
mbs, mbs_sr = librosa.load("kmk_mister_blue_sky.wav", sr=None, mono=True)

D = np.abs(librosa.stft(mbs, n_fft=512, hop_length=128))
plt.figure()
ld.specshow(librosa.amplitude_to_db(D, ref=np.max),
            sr=mbs_sr, hop_length=128, x_axis='time', y_axis='hz')
plt.colorbar(format='%+2.0f dB')
plt.title('"Mister Blue Sky" spectrogram')
plt.show()

##### Question 6 [30 points]

In this question we will create a song based on the spoken sentence you recorded. You will choose the melody by creating a sequence of pitches that change slowly over time.
Write a function that does the following:

1. Divide the speech signal into short slices (frames) of 512 samples with 50% overlap
2. For each speech segment compute formants by converting lpc_to_formants (F,Fb)
3. Choose a pitch (f0) for each segment
4. Using the voca function, create a speech sound (vow) from an excitation (ex) with that pitch
4. Overlap and add the sound segments with cross-fade window to create one long sound file 

You are free to alter the durations of the segments and choice of notes for the melody. 
Note that the notes should be relatively long (f0 should not change very often).

Cross-fade between segments can be done by applying a traingular (numpy.bartlett) or raised cosine (numpy.hanning) window to each segment before.

In [ ]:
# Read the required mono speech file.
# Use the same "Mister Blue Sky" recording you analyzed in Question 5 (mono).
sr, wave = wavfile.read("kmk_mister_blue_sky.wav")
assert wave.ndim == 1
wave = wave.astype(np.float64)

frame_len = 512
hop_length = frame_len // 2
num_frames = 1 + int(
    np.ceil(max(0, len(wave) - frame_len) / hop_length)
)
padded_len = (num_frames - 1) * hop_length + frame_len
padded_wave = np.pad(wave, (0, padded_len - len(wave)))

# Periodic Hann is COLA-compatible at 50% overlap.
window = scipy.signal.windows.hann(frame_len, sym=False)
vocode = np.zeros(padded_len, dtype=np.float64)

lpc_order = 10
# Melody (Hz); notes are held long so f0 changes slowly across the whole file.
melody = np.array([220.00, 246.94, 293.66, 329.63, 293.66, 246.94, 220.00])  # A3 B3 D4 E4 D4 B3 A3
frames_per_note = max(1, num_frames // len(melody))

for frame_index, start in enumerate(
    range(0, padded_len - frame_len + 1, hop_length)
):
    wave_slice = padded_wave[start:start + frame_len]

    ### Your Code Here
    # 1. LPC of this speech frame -> formants (F, Fb)
    if np.any(wave_slice):
        a = librosa.core.lpc(wave_slice, order=lpc_order)
        F, Fb = lpc_to_formants(a, sr)
    else:
        F, Fb = np.array([]), np.array([])   # silent (padding) frame

    # 2. Pick a pitch (f0) for this segment from the slowly-changing melody
    f0 = float(melody[min(frame_index // frames_per_note, len(melody) - 1)])

    # 3. Build an excitation exactly frame_len long (dur = frame_len / sr)
    ex = excitation(f0, ji, frame_len / sr, sample_rate=sr)
    assert len(ex) == frame_len

    vow, B, A = voca(ex, F, Fb, sample_rate=sr)
    assert len(vow) == frame_len
    vocode[start:start + frame_len] += vow * window

vocode = vocode[:len(wave)]
# Normalize to avoid clipping, then write the vocoded song.
vocode = vocode / (np.max(np.abs(vocode)) + 1e-12)
wavfile.write(
    "mister_blue_sky_vocoded.wav",
    sr,
    vocode.astype(np.float32),
)

##### Question 7 [10 points]

Why did we use overlapping windows for vocoder? 

Because building the song frame-by-frame and gluing the frames end-to-end would create audible clicks and pops — overlapping windows fix that with a smooth cross-fade.

- Each frame is synthesized **independently** — its own formants, its own pitch (`f0`). Laid right next to each other, the waveform would **jump discontinuously** at every seam (frame 1 ends at some value, frame 2 starts at a different one). Sharp discontinuities are heard as clicks/pops, so the audio would sound choppy.
- To avoid that, each frame is multiplied by a **window that tapers to zero at both edges** (the Hann window here, or a Bartlett/triangular one). Consecutive frames **overlap by 50% and are added together** — the **overlap-add** method. Where one frame fades out, the next fades in, giving a smooth **cross-fade** instead of a hard cut.
- This also matches how speech actually behaves: the vocal tract and pitch change *gradually*, not in sudden jumps. Overlap-add lets the parameters drift smoothly across the whole file.
- Using a **Hann window at 50% overlap** specifically satisfies the **Constant Overlap-Add (COLA)** condition: sliding those windows by half a frame and summing them gives a constant. So the cross-fades don't make the volume pulse up and down — the loudness stays even. A badly matched window/overlap would cause an amplitude "wobble" at the frame rate.

So overlapping windows do two jobs at once: they **smooth the transitions between frames so there are no clicks**, and (with the right window + overlap) they **keep the overall amplitude constant** so the reconstructed song is continuous and even.